# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

Building the Baseline Score First

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

feature_frame = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) AS feat_clicks,
            AVG(CASE WHEN period = 'first_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_first_half,
            AVG(CASE WHEN period = 'second_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_second_half,
            SUM(CASE WHEN period = 'first_half' THEN 1 ELSE 0 END) AS feat_days_active,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) * 1.0
                / NULLIF(SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END), 0) AS feat_ctr
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_clicks,
        f.position_first_half,
        f.position_second_half,
        (f.position_second_half - f.position_first_half) AS position_change,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half - l.first_half) * 1.0 / NULLIF(l.first_half, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half > 0
      AND f.position_first_half IS NOT NULL
      AND f.position_second_half IS NOT NULL
    ORDER BY f.content_hash_id, f.client_hash_id
""").df()

# Belt-and-suspenders: also reset the index after sorting, so row positions are fully deterministic
feature_frame = feature_frame.reset_index(drop=True)

print(feature_frame.shape)

print("\n=== Feature Frame ===")
print(feature_frame.shape)
feature_frame.head()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(139747, 10)

=== Feature Frame ===
(139747, 10)


,content_hash_id,client_hash_id,feat_impressions,feat_clicks,position_first_half,position_second_half,position_change,feat_days_active,feat_ctr,declining_flag
0,content_000005d4ced12088,client_9958f0a7ae1df715,23.0,0.0,72.101852,73.306667,1.204815,9.0,0.000000,False
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,22.0,0.0,12.600000,6.600000,-6.000000,12.0,0.000000,False
2,content_0000cd28fbda69f3,client_3ffa76342f366962,11.0,0.0,4.062500,4.553333,0.490833,8.0,0.000000,False
3,content_00014efc121d911d,client_08a6a72ff48e62c0,53.0,0.0,6.768864,4.688095,-2.080769,14.0,0.000000,False
4,content_000184dde41afe75,client_62f4a7e64f5e0096,2405.0,8.0,3.682934,3.477619,-0.205315,15.0,0.003326,False


In [17]:
# Rebuild the baseline rule/score exactly as in w04
feature_frame['high_volume'] = (feature_frame['feat_impressions'] >= 500).astype(int)
feature_frame['position_slipped'] = (feature_frame['position_change'] > 2).astype(int)
feature_frame['score'] = (
    feature_frame['high_volume']
    * feature_frame['position_slipped']
    * feature_frame['feat_impressions']
)

print(feature_frame.shape)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

baseline_precision_20 = precision_at_k(feature_frame['score'].values, feature_frame['declining_flag'].values, 20)
baseline_precision_50 = precision_at_k(feature_frame['score'].values, feature_frame['declining_flag'].values, 50)
base_rate = feature_frame['declining_flag'].mean()

print(f"Baseline precision@20: {baseline_precision_20:.3f}")
print(f"Baseline precision@50: {baseline_precision_50:.3f}")
print(f"Base rate (overall decline rate): {base_rate:.3f}")

print(f"{len(feature_frame):,} rows")

(139747, 13)
Baseline precision@20: 0.500
Baseline precision@50: 0.400
Base rate (overall decline rate): 0.278
139,747 rows


Note: Baseline precision@50 = 0.400, against a base rate of 0.278. The rule performs above chance, but with real room for improvement and consistent with the ~50% rule/label disagreement found in the Week-4 top-20 review.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Decision Tree then Random Forest

Why: it fits my lane as i have a binary future yes/no answer

`declining_flag = True/False`

Reason for choosing Decsision Tree is to have a readable model at hand.
Decision Tree first (readable, matches my baseline exploration style) then Random Forest (usually stronger, still interpretable via feature importance).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Im going to be grouping based upon clients, because the same client on both train and test splits will cause issues like the model learning client specific patterns and cheating.

this might give us better precision but it would not work on unseen or new data.

Split is grouped by client_hash_id to prevent leakage of client-specific patterns across train/test. Because client sizes are highly uneven (the largest client alone is 18% of all rows; the top 4 clients together exceed 54%), the resulting split lands at roughly 92/8 rather than the intended 80/20. This is an expected consequence of grouping on an unbalanced panel, not an error.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Reuse the feature_frame already built earlier in this notebook
feature_cols = ['feat_impressions', 'feat_clicks', 'position_first_half',
                'feat_days_active', 'feat_ctr']

X = feature_frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = feature_frame['declining_flag']
groups = feature_frame['client_hash_id']  # used ONLY for splitting, never as a feature

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]


In [20]:
# --- Checks ---
print(f"Train size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")


Train size: 128111 (91.7%)
Test size: 11636 (8.3%)


In [21]:
overlap = set(groups_train.unique()) & set(groups_test.unique())
print(f"Clients appearing in BOTH train and test: {len(overlap)}")


Clients appearing in BOTH train and test: 0


In [22]:
print(f"\nTrain decline rate: {y_train.mean():.3f}")
print(f"Test decline rate: {y_test.mean():.3f}")


Train decline rate: 0.276
Test decline rate: 0.301


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.tree import DecisionTreeClassifier, export_text
import pandas as pd


In [24]:
# Final split, locked in
#splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Final model
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

tree_scores_test = tree.predict_proba(X_test)[:, 1]
tree_precision_50 = precision_at_k(tree_scores_test, y_test.values, 50)

# Baseline, evaluated on the SAME test set for a fair comparison
baseline_scores_test = feature_frame.loc[X_test.index, 'score'].values
baseline_precision_50_test = precision_at_k(baseline_scores_test, y_test.values, 50)

base_rate_test = y_test.mean()

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=4, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

rf_scores_test = rf.predict_proba(X_test)[:, 1]
rf_precision_50 = precision_at_k(rf_scores_test, y_test.values, 50)

# Precision at multiple k values for all methods
k_values = [20, 50]

methods_scores = {
    'Baseline rule': baseline_scores_test,
    'Decision Tree': tree_scores_test,
    'Random Forest': rf_scores_test,
}

rows = []
for method, scores in methods_scores.items():
    row = {'method': method}
    for k in k_values:
        row[f'precision_at_{k}'] = precision_at_k(scores, y_test.values, k)
    rows.append(row)

# Base rate is k-independent, but shown for reference at each k
base_row = {'method': 'Base rate'}
for k in k_values:
    base_row[f'precision_at_{k}'] = base_rate_test
rows.append(base_row)

comparison_table = pd.DataFrame(rows)
print(comparison_table)

          method  precision_at_20  precision_at_50
0  Baseline rule         0.250000         0.240000
1  Decision Tree         0.400000         0.520000
2  Random Forest         0.500000         0.460000
3      Base rate         0.301134         0.301134


The model seems to beat the baseline rule.
Interesting observation here is definitly the RandomForest model achieving less precision@50 compared to the Decision Tree model.

Notes:



*   Depth 4 was explored as a recall to the `your_first_readable_model.ipynb`, i had stated there that a depth 4 tree was still readable.
*   The First Feature set included `position_second_half` and `position_change`, the model showed better precision@50 but due to obvious leakage possibilities it was dropped from the feature set.
* The feature set at the moment ensures that the model does not have access to any future information regarding the page like the data from the second_half of the month. This will replicate the precision in real world unseen data

| Method | Precision@20 | Precision@50
|---|---|---|
| Baseline (Week 4 rule: `position_slipped x high_volume x feat_impressions`) | 0.25 | 0.24
| Decision Tree | 0.40 | 0.52
| Random Forest | 0.50 | 0.46

The model clearly beat the baseline precision at both 20 and 50.

Note: a pure leakage issue was occured during training because of the feature addition, position_change and second_half_position was also added as a part of the feature set but that clearly would mean that the model would have access to the data which will clearly not be present during real world scenario.
the precision now is much more true and restricts the model from looking at future data.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**what does it lean on:**

The Decision Tree Model as shown here relies heavily on `feat_impressions` = 36.59%. Part of this is because our label `declining_flag` is formulated purely using impressions change. Position never enters this calculation at all. This is your ground-truth label, matching FlyRank's own trend_pct <= -20 cutoff (which you confirmed empirically was also impressions-based, back in ML-03).
our `score` label uses the `feat_impression` as a multiplier alongside `high_volume` and `position_slipped`.
feat_impressions is also the first cutoff point at depth 1 where the model reasons to diffrentiate based upon if the page had any significant traffic or none at all.

`feat_days_active` is also another label the model relies on.

**where is the model wrong:**

The **model** is predicting pages as declining when `feat_impressions > 1.5`, `clicks = 0` and `feat_days_active <= 13.5`. These rows share the exact same predicted probability `(0.515502)`, which means they land in the exact same leaf of the tree, not just a similar bucket. In plain
terms: the model is treating "low activity + zero clicks + few active days" as its criteria for "declining" but this really just describes a thin, low-traffic page, not necessarily one that's actually losing impressions.

There's also a clear bias toward predicting decline: 8 of my 10 sampled wrong cases are false positives (predicted True, actually False), versus only 2 false negatives. The model appears to over-flag sparse, low-data pages as declining rather than missing genuine decliners.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)


print(export_text(tree, feature_names=feature_cols))

test_results = X_test.copy()
test_results['true_label'] = y_test.values
test_results['predicted_prob'] = tree_scores_test
test_results['predicted_label'] = tree_scores_test >= 0.5

               feature  importance
0     feat_impressions    0.365919
3     feat_days_active    0.247968
4             feat_ctr    0.193091
2  position_first_half    0.108842
1          feat_clicks    0.084179
|--- feat_impressions <= 1.50
|   |--- class: False
|--- feat_impressions >  1.50
|   |--- feat_ctr <= 0.00
|   |   |--- feat_days_active <= 13.50
|   |   |   |--- feat_impressions <= 117.50
|   |   |   |   |--- class: True
|   |   |   |--- feat_impressions >  117.50
|   |   |   |   |--- class: False
|   |   |--- feat_days_active >  13.50
|   |   |   |--- feat_impressions <= 185.50
|   |   |   |   |--- class: True
|   |   |   |--- feat_impressions >  185.50
|   |   |   |   |--- class: True
|   |--- feat_ctr >  0.00
|   |   |--- feat_clicks <= 3.50
|   |   |   |--- feat_impressions <= 33.50
|   |   |   |   |--- class: True
|   |   |   |--- feat_impressions >  33.50
|   |   |   |   |--- class: False
|   |   |--- feat_clicks >  3.50
|   |   |   |--- position_first_half <= 18.12
|   

In [29]:
wrong_cases = test_results[test_results['true_label'] != test_results['predicted_label']]
print(wrong_cases.shape)
wrong_cases.sample(10, random_state=42)

(5880, 8)


,feat_impressions,feat_clicks,position_first_half,feat_days_active,feat_ctr,true_label,predicted_prob,predicted_label
127787,12.0,0.0,52.250000,4.0,0.000000,False,0.515502,True
126181,338.0,2.0,7.566951,15.0,0.005917,True,0.447109,False
4562,1306.0,3.0,10.146332,15.0,0.002297,False,0.608819,True
78205,16.0,0.0,42.283333,4.0,0.000000,False,0.515502,True
83956,9.0,0.0,5.500000,8.0,0.000000,False,0.515502,True
132980,8.0,0.0,19.166667,4.0,0.000000,False,0.515502,True
56100,32.0,0.0,37.020833,11.0,0.000000,False,0.515502,True
39363,145.0,0.0,9.420438,15.0,0.000000,False,0.537188,True
136942,1653.0,0.0,4.899161,15.0,0.000000,False,0.608819,True
52871,2.0,0.0,8.500000,2.0,0.000000,False,0.515502,True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.